<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/tapvidmv/verify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID ground truth — does the export hold up?

This reads the **exported release** under `tapvidmv/data/release/`, not the pipeline's
intermediate state, so what it checks is the thing that ships.

DROID's rig is three cameras: view 0 rides on the wrist and travels with the arm,
views 1 and 2 are fixed on the table. One set of 3D tracks in world space is
projected into all three. That is what makes the ground truth checkable — the
same physical point has to land on the same physical thing in every view, and
the depth maps from two cameras have to agree about where that thing is.

Work down the notebook and the question sharpens:

| Section | Asks |
|---|---|
| 2 | Is the rig what we think it is — one camera moving, two still? |
| 3 | Do the 2D tracks stay glued to their surfaces? |
| 4 | Do the 3D tracks lie *on* the depth cloud, or float above it? |
| 5 | Do the views agree about what is visible, and where the queries live? |
| 6 | Is the depth sane inside the 2 m workspace, and does the wrist mask fit? |
| 7 | Read a point's depth in one view, reproject into another — how far off? |
| 8 | All fifty at once: which episodes need opening? |

---
## 0. Setup

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
  subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "https://github.com/yangyi02/droid.git", "/content/droid"],
    check=True,
  )
  os.chdir("/content/droid/tapvidmv")
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapy", "plotly"], check=True)

HERE = os.getcwd()
REPO = os.path.dirname(HERE)
sys.path[:0] = [REPO, HERE]

import matplotlib.pyplot as plt
import mediapy
import numpy as np

import release
import viz

plt.rcParams["figure.dpi"] = 110

DATA_ROOT = release.find_dataset()
EPISODES = release.episode_names(DATA_ROOT)
MAX_DEPTH_M = release.MAX_DEPTH_M

print(f"{len(EPISODES)} episodes under {DATA_ROOT}")
print("\n".join(f"  [{index:2d}] {name}" for index, name in enumerate(EPISODES)))

---
## 1. Pick an episode

In [ ]:
EPISODE_INDEX = 0
NUM_TRACKS = 24
TRAIL = 12

episode = release.load_episode(EPISODES[EPISODE_INDEX], DATA_ROOT)
FRAME = episode.num_frames // 2
track_ids = viz.pick_tracks(episode.visibility.transpose(2, 0, 1), NUM_TRACKS, frame=FRAME, require_views=2)
colors = viz.track_colors(track_ids)

print(f"{episode.name}")
print(f"  {episode.num_frames} frames, {episode.num_tracks} tracks, {episode.num_views} views")
print(f"  showing {len(track_ids)} tracks at frame {FRAME}")

---
## 2. The rig

DROID's signature: one camera on the wrist, two bolted to the table. The wrist
view is the hard one — it moves, so its extrinsics change every frame, and any
error in the arm's forward kinematics shows up there first.

In [ ]:
rows = []
for view, data in enumerate(episode.views):
  height, width = data.image_hw
  rows.append(
    f"  view {view}  {data.kind:24s} {width}x{height}  "
    f"visible {100 * data.visibility.mean():5.1f}%  "
    f"depth {'yes' if data.depth(0) is not None else 'no ':3s}  "
    f"mask {'yes' if data.foreground_mask(0) is not None else 'no'}"
  )
print(f"{episode.name}\n" + "\n".join(rows))

In [ ]:
figure = plt.figure(figsize=(11, 4.4))
ax = figure.add_subplot(1, 2, 1, projection="3d")
for view, data in enumerate(episode.views):
  centers = data.centers
  color = viz.view_color(view) / 255.0
  ax.plot(*centers.T, color=color, lw=2, label=f"view {view} ({data.kind.split(',')[0]})")
  ax.scatter(*centers[0], color=color, s=40, marker="D", depthshade=False)
cloud = episode.tracks_xyz.reshape(-1, 3)
ax.scatter(*cloud[::97].T, color="#b8b8b4", s=1, depthshade=False)
ax.set_box_aspect((1, 1, 0.6))
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("camera paths and the tracked points", fontsize=10)

ax = figure.add_subplot(1, 2, 2)
for view, data in enumerate(episode.views):
  offsets = np.linalg.norm(data.centers - data.centers[0], axis=-1)
  ax.plot(offsets, color=viz.view_color(view) / 255.0, lw=2, label=f"view {view}")
ax.set_xlabel("frame"); ax.set_ylabel("camera travel from frame 0 (m)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
ax.set_title("which cameras move", fontsize=10, loc="left")
figure.tight_layout()

---
## 3. The 2D ground truth

### 3.1 One frame, every view

The same 3D points, the same colours, in every camera at the same instant. A
colour that lands on the gripper in one view and on the table in another is a
projection error — the single most useful thing to look for here.

In [ ]:
panel = viz.all_views_panel(episode, FRAME, track_ids, colors=colors, cell_width=640, trail=TRAIL)
plt.figure(figsize=(17, 17 * panel.shape[0] / panel.shape[1]))
plt.imshow(panel); plt.axis("off"); plt.tight_layout()

### 3.2 The whole episode

Watch whether each dot stays glued to its physical point. A dot that slides
across a surface, or that stays filled while the arm passes in front of it, is
wrong in a way no threshold in `select_episodes.py` can see.

In [ ]:
video = viz.episode_video(episode, track_ids, colors=colors, num_frames=120, cell_width=440, trail=TRAIL)
mediapy.show_video(video, fps=12, title=f"{episode.name} - 2D ground truth")

### 3.3 One track, every camera

With 24 dots on screen it is hard to tell which is which. This takes a single
track and crops every view around it: **green border = that view calls it
visible, red = occluded**, with the depth it sits at.

In [ ]:
visible_counts = episode.visibility[FRAME][track_ids].sum(axis=-1)
FOCUS_TRACK = int(track_ids[int(np.argmax(visible_counts))])
strip = viz.montage(viz.crossview_patches(episode, FOCUS_TRACK, FRAME), columns=episode.num_views, cell_width=240)
plt.figure(figsize=(11, 11 * strip.shape[0] / strip.shape[1]))
plt.imshow(strip); plt.axis("off")
plt.title(f"track {FOCUS_TRACK} at frame {FRAME}", fontsize=10)
plt.tight_layout()

---
## 4. The 3D ground truth

### 4.1 Tracks against the depth cloud

The clouds from the two fixed cameras are the reference surface. Correct 3D
tracks lie **on** it — a curve floating above the table, or sunk into it, is the
failure this view exists to catch. Rotate it.

In [ ]:
figure = viz.plot_tracks_3d(episode, track_ids, frame=FRAME, colors=colors)
figure.show()

### 4.2 3D and 2D side by side

The same instant and the same colours in both rows: the camera images on top,
the 3D tracks below from three angles. If a point looks right in 2D but wrong in
3D, the depth behind it is the suspect, not the track.

In [ ]:
AZIMUTHS = (-60, 10, 80)
figure = plt.figure(figsize=(16, 8.6))
for column in range(episode.num_views):
  ax = figure.add_subplot(2, episode.num_views, column + 1)
  ax.imshow(viz.view_panel(episode, column, FRAME, track_ids, colors=colors, trail=TRAIL, label=False, width=560))
  ax.set_axis_off()
  ax.set_title(f"view {column} ({episode.views[column].kind})", fontsize=9)
for column, azim in enumerate(AZIMUTHS[: episode.num_views]):
  ax = figure.add_subplot(2, episode.num_views, episode.num_views + column + 1, projection="3d")
  viz.tracks_3d_axes(ax, episode, track_ids, colors=colors, azim=azim)
  ax.set_title(f"azim {azim}", fontsize=9)
figure.tight_layout()

---
## 5. Multi-view structure

### 5.1 Co-visibility

The diagonal is each view's own visible rate; entry `(i, j)` is the fraction of
observations both views call visible. The wrist view sees less — it is close in
and the arm occludes itself — and a benchmark scored across views lives on the
off-diagonal.

In [ ]:
covis = viz.covisibility_matrix(episode.visibility)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6), width_ratios=[1, 1.7])

image = axes[0].imshow(covis, vmin=0, vmax=covis.max(), cmap="Blues")
for i in range(covis.shape[0]):
  for j in range(covis.shape[1]):
    axes[0].text(j, i, f"{covis[i, j]:.2f}", ha="center", va="center",
                 color="white" if covis[i, j] > covis.max() * 0.6 else "#333333", fontsize=9)
axes[0].set_xticks(range(episode.num_views)); axes[0].set_yticks(range(episode.num_views))
axes[0].set_title("co-visibility", fontsize=10, loc="left")

per_track = episode.visibility.any(axis=0).sum(axis=-1)
axes[1].hist(per_track, bins=np.arange(episode.num_views + 2) - 0.5, color="#2a78d6", linewidth=0)
axes[1].set_xticks(range(episode.num_views + 1))
axes[1].set_xlabel("views that ever see the track"); axes[1].set_ylabel("tracks")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].set_title("how many views see each track", fontsize=10, loc="left")
figure.tight_layout()

### 5.2 Where the queries live

The fourth column of `queries_xytv` is the **query view** — the view a point was
seeded in, on that view's exact pixel at t=0. Scoring splits on it: predicting a
point in the view it was queried from is a different problem from predicting it
in a view that never saw it start.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.4), width_ratios=[1, 1.6])
counts = np.bincount(episode.query_v, minlength=episode.num_views)
axes[0].bar(range(episode.num_views), counts,
            color=[viz.view_color(v) / 255.0 for v in range(episode.num_views)], width=0.62)
for view, count in enumerate(counts):
  axes[0].text(view, count, str(count), ha="center", va="bottom", fontsize=9, color="#52514e")
axes[0].set_xticks(range(episode.num_views))
axes[0].set_xlabel("query view"); axes[0].set_ylabel("tracks")
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].set_title("queries per view", fontsize=10, loc="left")

QUERY_VIEW = int(counts.argmax())
owned = np.flatnonzero(episode.query_v == QUERY_VIEW)
axes[1].imshow(viz.view_panel(episode, QUERY_VIEW, 0, owned[:: max(1, len(owned) // 80)], label=False, width=640))
axes[1].set_axis_off()
axes[1].set_title(f"view {QUERY_VIEW} at t=0, the points it seeded", fontsize=10, loc="left")
figure.tight_layout()

---
## 6. Depth, the 2 m workspace, and the wrist mask

Three DROID-specific things at once. Stereo depth is noisy and partial — expect
holes. Everything past **2 m** is off the table and irrelevant, so the tracking
depth is capped there. And `foreground_mask.npy` exists only for the wrist view,
where the gripper fills much of the frame.

In [ ]:
figure, axes = plt.subplots(episode.num_views, 3, figsize=(14, 3.1 * episode.num_views))
axes = np.atleast_2d(axes)
for view, data in enumerate(episode.views):
  image = data.image(FRAME)
  depth = data.depth(FRAME)
  axes[view, 0].imshow(image)
  axes[view, 0].set_title(f"view {view} ({data.kind})", fontsize=9, loc="left")

  if depth is None:
    axes[view, 1].text(0.5, 0.5, "no depth.npy", ha="center", va="center")
    axes[view, 2].text(0.5, 0.5, "-", ha="center", va="center")
  else:
    capped = viz.cap_depth(depth)
    coverage = 100 * (capped > 0).mean()
    axes[view, 1].imshow(viz.colorize_depth(capped))
    axes[view, 1].set_title(f"depth capped at {MAX_DEPTH_M:g} m - {coverage:.0f}% of pixels kept", fontsize=9, loc="left")

    mask = data.foreground_mask(FRAME)
    if mask is None:
      inside = np.isfinite(depth) & (depth > 0) & (depth <= MAX_DEPTH_M)
      axes[view, 2].imshow(viz.overlay_mask(image, ~inside, color=(40, 40, 40)))
      axes[view, 2].set_title("grey = outside the workspace or no depth", fontsize=9, loc="left")
    else:
      axes[view, 2].imshow(viz.overlay_mask(image, mask.astype(bool)))
      axes[view, 2].set_title(f"foreground mask - {100 * mask.mean():.0f}% of the frame", fontsize=9, loc="left")
for ax in axes.ravel():
  ax.set_axis_off()
figure.tight_layout()

---
## 7. Is the ground truth self-consistent across views?

The sharpest check in the notebook.

Take a point both views can see. Read the **source view's depth map** at the
ground-truth pixel, unproject that reading into the world, and project it into
the **other** view. If the tracks and the depth agree, the two land on the same
pixel. The gap is in the units the benchmark is scored in.

In [ ]:
PAIR_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]

frames = np.unique(np.linspace(0, episode.num_frames - 1, 24).astype(int))
pairs = [(i, j) for i in range(episode.num_views) for j in range(i + 1, episode.num_views)]
results = {
  pair: np.concatenate([viz.crossview_gap_px(episode, *pair, frames), viz.crossview_gap_px(episode, *pair[::-1], frames)])
  for pair in pairs
}

span = np.percentile(np.concatenate([g for g in results.values() if len(g)]), 98)
figure, ax = plt.subplots(figsize=(9, 3.6))
for index, ((source, target), gaps) in enumerate(results.items()):
  if not len(gaps):
    continue
  ax.hist(gaps, bins=60, range=(0, span), histtype="step", linewidth=1.7,
          color=PAIR_COLORS[index % len(PAIR_COLORS)],
          label=f"views {source} and {target}   median {np.median(gaps):.1f} px   p95 {np.percentile(gaps, 95):.1f} px")
ax.set_xlabel("reprojection gap (px)"); ax.set_ylabel("observations")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"depth read in one view, reprojected into the other - {len(frames)} frames", fontsize=10, loc="left")
figure.tight_layout()

---
## 8. Every episode

The sweep. Anything that looks wrong here is worth opening in section 1 by
setting `EPISODE_INDEX`.

In [ ]:
SWEEP_EPISODES = EPISODES
SWEEP_TRACKS = 20
SWEEP_FRAMES = 40


def sweep_ids(other):
  return viz.pick_tracks(other.visibility.transpose(2, 0, 1), SWEEP_TRACKS, frame=other.num_frames // 2, require_views=2)


print(f"{len(SWEEP_EPISODES)} episodes x {SWEEP_TRACKS} tracks x {episode.num_views} views")

### 8.1 One frame per episode

In [ ]:
tiles = []
for index, name in enumerate(SWEEP_EPISODES):
  other = release.load_episode(name, DATA_ROOT)
  middle = other.num_frames // 2
  panel = viz.all_views_panel(other, middle, sweep_ids(other), cell_width=300, trail=8, label=False)
  tiles.append(viz.label_panel(panel, f"[{index}] {name}"))

sheet = viz.montage(tiles, columns=1, cell_width=940)
plt.figure(figsize=(15, 15 * sheet.shape[0] / sheet.shape[1]))
plt.imshow(sheet); plt.axis("off"); plt.tight_layout()

### 8.2 Every episode as a video

One player per episode, ground truth burned in. Episodes differ in length, so
each is resampled to `SWEEP_FRAMES` frames.

In [ ]:
videos = {}
for index, name in enumerate(SWEEP_EPISODES):
  other = release.load_episode(name, DATA_ROOT)
  videos[f"[{index}] {name.split('+')[0]}...{name[-13:]}"] = viz.episode_video(
    other, sweep_ids(other), num_frames=SWEEP_FRAMES, cell_width=300, trail=8, label=False
  )

print(f"{len(videos)} videos, {sum(v.nbytes for v in videos.values()) / 1e6:.0f} MB")
mediapy.show_videos(videos, fps=10, columns=2, height=190)

### 8.3 All the 3D tracks on one sheet

Each panel is one episode's complete set of 3D tracks plus its camera paths.
Diamonds are fixed cameras, lines are the wrist. Looking for: trajectories that
explode, collapse to a point, or wander off away from the cameras.

In [ ]:
SHEET_TRACKS = 80
COLUMNS = 5

rows_count = int(np.ceil(len(SWEEP_EPISODES) / COLUMNS))
figure = plt.figure(figsize=(3.1 * COLUMNS, 3.1 * rows_count))
for index, name in enumerate(SWEEP_EPISODES):
  other = release.load_episode(name, DATA_ROOT)
  ax = figure.add_subplot(rows_count, COLUMNS, index + 1, projection="3d")
  ids = viz.pick_tracks(other.visibility.transpose(2, 0, 1), SHEET_TRACKS, require_views=1)
  viz.tracks_3d_axes(ax, other, ids, linewidth=0.6)
  ax.set_title(f"[{index}] {name.split('+')[0]}", fontsize=8)
figure.tight_layout()

---
## 9. Save

Per-episode MP4s of the 2D ground truth, so they can be reviewed outside the
notebook. Set `SAVE = True` to write them.

In [ ]:
SAVE = False
OUT_DIR = os.path.join(HERE, "groundtruth_review")

if SAVE:
  os.makedirs(OUT_DIR, exist_ok=True)
  for name in SWEEP_EPISODES:
    other = release.load_episode(name, DATA_ROOT)
    clip = viz.episode_video(other, sweep_ids(other), num_frames=SWEEP_FRAMES, cell_width=380, trail=8)
    path = os.path.join(OUT_DIR, name.replace("+", "_") + ".mp4")
    mediapy.write_video(path, clip, fps=10)
    print(f"  {path}")
  print(f"{len(SWEEP_EPISODES)} clips -> {OUT_DIR}")
else:
  print("SAVE = False")